In [ ]:
import pandas as pd
import os

In [ ]:
def combinar_h_v(df):
    """
    Combina pares de columnas que empiezan por 'h' y 'v' en un DataFrame,
    creando nuevas columnas con los valores combinados. Además, devuelve
    una lista de las columnas creadas.

    Args:
        df (pd.DataFrame): DataFrame original.

    Returns:
        pd.DataFrame: DataFrame con columnas combinadas añadidas.
        list: Lista de nombres de las columnas creadas.
    """
    columnas_h = [col for col in df.columns if col.startswith('h')]
    columnas_v = [col for col in df.columns if col.startswith('v')]
    
    columnas_creadas = []  # Lista para almacenar nombres de las columnas creadas

    for col_h, col_v in zip(columnas_h, columnas_v):
        nueva_columna = f"{col_h}_{col_v}"  # Nombre de la nueva columna
        df[nueva_columna] = df[col_h].astype(str) + " " + df[col_v].astype(str)
        columnas_creadas.append(nueva_columna)  # Agregar el nombre de la columna a la lista
    
    return df, columnas_creadas

In [ ]:
def formato_df(df):
    """
    Formatea un DataFrame con datos de calidad del aire, asegurando un formato homogéneo y estructurado.

    Pasos realizados:
    1. Convierte las columnas "provincia", "municipio", "estacion", "magnitud", "ano", "mes" y "dia" a tipo string.
    2. Rellena los valores de mes y día para que tengan siempre dos dígitos.
    3. Crea una nueva columna "fecha" combinando año, mes y día.
    4. Une las columnas de hora y validación mediante la función `combinar_h_v()`.
    5. Transforma las columnas de hora en filas mediante `pd.melt()`.
    6. Separa la validación del valor y deja únicamente la hora en la columna "hora".
    7. Ajusta el valor "24" en la columna "hora" a "23:59" y añade ":00" en las demás horas.
    8. Crea una columna "fecha_hora_f" en formato datetime.
    9. Extrae el estado de validación de la medida desde la columna "valor".
    10. Corrige los valores con comas decimales, reemplazando `,` por `.` en la columna "valor".
    11. Convierte la columna "valor" a tipo float para realizar operaciones numéricas.
    12. Crea un identificador único "id_medida" para cada observación.
    13. Une "provincia", "municipio" y "estacion" en "codigo_estacion" con formato estandarizado.
    14. Elimina las filas con valores nulos.

    Args:
        df : pd.DataFrame
            DataFrame con los datos de calidad del aire.

    Returns:
        pd.DataFrame
            DataFrame formateado con la estructura correcta.
    """
    # Pasamos las columnas de "provincia", "municipio", "estacion" y "magnitud" a str porque no vamos a operar con esos números
    df[["provincia", "municipio", "estacion", "magnitud"]] = df[["provincia", "municipio", "estacion", "magnitud"]].astype(str)
    # Nos aseguramos de que las columnas de fecha sean str
    df[["ano", "mes", "dia"]] = df[["ano", "mes", "dia"]].astype(str)

    # Rellenamos mes y día para que siempre tengan 2 dígitos
    df["mes"] = df["mes"].str.zfill(2)
    df["dia"] = df["dia"].str.zfill(2)

    # Creamos la columna de fecha
    df["fecha"] = df["ano"] + "-" + df["mes"] + "-" + df["dia"]

    # Combinamos las columnas de hora y validación
    df, columnas_creadas = combinar_h_v(df)

    # Transformamos el DataFrame para que las columnas de hora queden en filas
    df = df.melt(
        id_vars=["provincia", "municipio", "estacion", "magnitud", "punto_muestreo", "fecha"],
        value_vars=columnas_creadas,
        var_name="hora",
        value_name="valor"
    )

    # Ahora separamos la letra que acompaña al valor para crear la columna de validación y dejamos solo la hora
    df["hora"] = df["hora"].str.split("_", expand=True)[0].str.extract("(\\d+)")[0]

    # Ajustamos el formato de la hora para casos especiales
    df["hora"] = df["hora"].apply(lambda x: '23:59' if x == '24' else f"{str(x)}:00")

    # Creamos una columna de fecha y hora con formato datetime
    df["fecha_hora_f"] = pd.to_datetime(df["fecha"].astype(str) + " " + df["hora"].astype(str))

    # Extraemos el estado de validación de la medida
    df["validacion"] = df["valor"].str.split(" ", expand=True)[1]
    df["valor"] = df["valor"].str.split(" ", expand=True)[0]

    # Corrige los valores con comas decimales antes de convertir a float
    df["valor"] = df["valor"].str.replace(",", ".")  # Reemplaza comas por puntos en decimales

    # Convertimos "valor" a float
    df["valor"] = df["valor"].astype(float)

    # Creamos un id único para cada medida
    df["id_medida"] = df["punto_muestreo"] + "_" + df["fecha"] + "_" + df["hora"] + "_" + df["validacion"]

    # Formamos el código de estación
    df["codigo_estacion"] = df["provincia"].astype(str).str.zfill(2) + \
                            df["municipio"].astype(str).str.zfill(3) + \
                            df["estacion"].astype(str).str.zfill(3)
    df = df.dropna()
    return df

In [ ]:
def unir_archivos(carpeta_entrada, patron_nombre, carpeta_salida, nombre_salida):
    """
    Une los archivos de una carpera según el nombre dado y los guarda como un único parquet en la carpeta de salida

    Parámetros:
        carpeta_entrada (str): ruta de la carpeta donde se encuentran los archivos que queremos unir.
        patron_nombre (str): patrón de nombre de los archivos a unir
        carpeta_salida (str): ruta de la carpeta donde se guardan los archivos unidos.
        nombre_salida (str): nombre del archivo de salida (sin extensión).

    Retorna:
        archivos: Lista con los nombres de los archivos unidos.
    """
    carpeta = carpeta_entrada
    archivos = [archivo for archivo in os.listdir(carpeta) if archivo.lower().startswith(patron_nombre.lower())]
    df_lista = [pd.read_csv(os.path.join(carpeta, archivo), sep=";", parse_dates = True, encoding="latin1", low_memory = False) for archivo in archivos]
    print (archivos)
    df_unido = pd.concat(df_lista, ignore_index=True)
    df_unido.to_parquet(f"{carpeta_salida}/{nombre_salida}.parquet", index=False)
    print(f"archivo {nombre_salida}.parquet creado en {carpeta_salida}")
    return df_unido

In [ ]:
def formato_df_madrid(df):
    """
    Formatea el DataFrame de Madrid para que tenga el mismo formato que el de la Comunidad de Madrid.

    Args:
        df (pd.DataFrame): DataFrame original de Madrid.

    Returns:
        pd.DataFrame: DataFrame formateado.
    """
   
    # eliminamos duplicados
    df.drop_duplicates(inplace=True)
    # eliminamos una columna con muchos nulos
    df = df.drop('ï»¿PROVINCIA', axis=1)
    # pasamos todos los encabezdos a minúsculas
    df.columns = df.columns.str.lower()
    # añadimos la columna de provincia al principio
    df.insert(0, "provincia", 28)
    return df

In [ ]:
def crear_df_medidas(df_madrid, df_cmadrid):
    df_madrid = formato_df_madrid(df_madrid)
    if df_madrid.columns.equals(df_cmadrid.columns):
        df_medidas = pd.concat([df_madrid, df_cmadrid], ignore_index=True)
        df_medidas = formato_df(df_medidas)
    return df_medidas

In [ ]:
from unidecode import unidecode

In [ ]:
def formato_contaminantes(df):
    """
    Formatea el DataFrame de contaminantes para que tenga el formato adecuado.

    Args:
        df (pd.DataFrame): DataFrame original de contaminantes.

    Returns:
        pd.DataFrame: DataFrame formateado.
    """
    # cambiamos el nombre de las columnas a snake_case y quitamos las tildes
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    df.columns = df.columns.map(unidecode)
    df.columns = df.columns.str.replace('\r', '', regex=True).str.strip()
    df.columns = df.columns.str.replace('\n', '', regex=True).str.strip()
    return df

In [ ]:
def tablas_contaminantes(df):
    """
    Extrae y organiza tablas relacionadas con técnicas de medición y contaminantes ambientales a partir de un DataFrame.

    Pasos realizados:
    1. Aplica la función `formato_contaminantes(df)` para garantizar la correcta estructura del DataFrame.
    2. Obtiene una tabla de técnicas de medición eliminando valores duplicados.
    3. Extrae información sobre contaminantes, incluyendo código, descripción, unidad y tipo de unidad.
    4. Filtra la tabla de contaminantes eliminando registros donde la magnitud sea "Ozono Quimioluminiscencia".

    Args:
        df : pd.DataFrame
            DataFrame con datos de calidad del aire.

    Returns:
        tuple: (df_tecnicas, df_contaminantes)
            - df_tecnicas (pd.DataFrame): DataFrame con las técnicas de medición sin valores duplicados.
            - df_contaminantes (pd.DataFrame): DataFrame con los contaminantes filtrados y estructurados.
    """
    formato_contaminantes(df)
    df_tecnicas = df[["codigo_tecnica_de_medida", "descripcion_tecnica_de_medida"]].drop_duplicates()
    df_contaminantes = df[['codigo_magnitud', 'descripcion_magnitud', 'unidad', 'descripcion_unidad']]
    df_contaminantes = df_contaminantes.drop(df_contaminantes[df_contaminantes["descripcion_magnitud"] == "Ozono Quimioluminiscencia"].index)
    return df_tecnicas, df_contaminantes

In [ ]:
def obtener_zonas(df):
    """
    Extrae y formatea la información de zonas de calidad del aire a partir de un DataFrame.

    Pasos realizados:
    1. Obtiene los valores únicos de la columna "zona_calidad_aire_descripcion".
    2. Elimina la palabra "Zona" de los nombres de las zonas.
    3. Separa el número de la zona y su descripción en dos columnas ("zona" y "descripcion").
    4. Elimina la columna original después de la separación.
    5. Añade manualmente la fila correspondiente a la zona de Madrid.

    Args:
        df : pd.DataFrame
            DataFrame con la columna "zona_calidad_aire_descripcion" que contiene la descripción de las zonas.

    Returns:
        pd.DataFrame
            DataFrame con dos columnas:
            - "zona": Número de la zona.
            - "descripcion": Nombre descriptivo de la zona de calidad del aire.
    """
    # cogemos los valores únicos de la columna "zona_calidad_aire_descripcion" de estaciones_cmadrid
    df_zonas = pd.DataFrame(df["zona_calidad_aire_descripcion"].unique())
    # eliminamos la palabra "Zona" de la columna
    df_zonas[0] = df_zonas[0].str.replace("Zona ", "")
    # separamos en dos columnas: zona con el número de la zona y descripcion con el nombre de la zona y eliminamos la columnna original
    df_zonas[["codigo_zona", "descripcion"]]= df_zonas[0].str.split(" ", n=1,  expand = True)
    df_zonas.drop(columns=[0], inplace=True)
    # añadimos la fila con la informacion de la zona de Madrid
    df_zonas.loc[6] = ["1", "Madrid"]
    return df_zonas

In [ ]:
def obtener_estaciones_y_municipios(estaciones_cmadrid, estaciones_madrid):
    """
    Procesa y unifica los datos de estaciones de calidad del aire de la Comunidad de Madrid y Madrid.

    - Extrae el código de zona de las estaciones de la Comunidad de Madrid.
    - Renombra columnas y elimina información redundante para homogeneizar el formato.
    - Agrupa los valores de benceno, tolueno y xileno en una nueva columna BTX.
    - Mapea los valores de contaminantes en Madrid al mismo formato que los de la Comunidad de Madrid.
    - Ajusta nombres de columnas y elimina aquellas innecesarias.
    - Convierte fechas al formato datetime.
    - Normaliza los nombres de columnas a snake_case.
    - Rellena valores nulos y ajusta el formato de coordenadas.
    - Convierte todas las columnas "analizador_" en tipo booleano.
    - Concatena los datos en un único DataFrame con estructura unificada.

    Parámetros:
        estaciones_cmadrid (pd.DataFrame): Datos de estaciones de la Comunidad de Madrid.
        estaciones_madrid (pd.DataFrame): Datos de estaciones de Madrid.

    Retorna:
        pd.DataFrame: DataFrame con las estaciones unificadas y procesadas.
    """
    # empezamos con estaciones_cmadrid
    # extraemos el número de zona de la columna "zona_calidad_aire_descripcion" en una nueva columna "codigo_zona"
    estaciones_cmadrid["codigo_zona"] = estaciones_cmadrid["zona_calidad_aire_descripcion"].str.extract(r"Zona (\d+)")
    # eliminamos la cadena "estacion_" de todas las columnas
    estaciones_cmadrid.columns = estaciones_cmadrid.columns.str.replace("estacion_", "")
    # las estaciones de Madrid unifican benceno, tolueno y xileno en la columna BTX, por lo que creamos una nueva columna "analizador_BTX" en estaciones_cmadrid
    # con el valor correspondiente a las 3 columnas individuales, si no fueran las 3 iguales, nos da un "Fallo"
    estaciones_cmadrid["analizador_BTX"] = estaciones_cmadrid.apply(lambda row: row["analizador_TOL"] if row["analizador_TOL"] == row["analizador_BEN"] == row["analizador_XIL"] else "Fallo", axis=1)
    # se pueden eliminar las columnas individuales que ahora son redundantes ["analizador_TOL", "analizador_BEN", "analizador_XIL"]
    # por dar la misma estructura que estaciones_madrid, duplicamos la columna "municipio" como "nombre_estacion"
    estaciones_cmadrid["nombre_estacion"] = estaciones_cmadrid["municipio"]
    # y cambiamos "direccion_postal" a "direccion"
    estaciones_cmadrid = estaciones_cmadrid.rename(columns={"direccion_postal":"direccion"})
    # convertimos "fecha_alta" a datetime
    estaciones_cmadrid["fecha_alta"] = pd.to_datetime(estaciones_cmadrid["fecha_alta"])
    # nos quedamos con el formato de coordendas GMS y eliminamos columnas innecesarias
    estaciones_cmadrid.drop(columns=['analizador_TOL', 'analizador_BEN', 'analizador_XIL', 'zona_calidad_aire_descripcion', 'coord_UTM_ETRS89_x', 'coord_UTM_ETRS89_y'], inplace=True)
    # cambiamos también estaciones_madrid
    # separamos nom_tipo en tipo area y tipo estacion
    estaciones_madrid[["tipo_area", "tipo_estacion"]] = estaciones_madrid["NOM_TIPO"].str.split(" ", expand=True)
    # mapeamos los valores de las columnas de contaminantes al mismo formato que los de cmadrid
    estaciones_madrid[['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']] = estaciones_madrid[['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']].map(lambda x: "Si" if x == "X" else "No")
    # renombramos las columnas de contaminantes para que tengan el mismo formato que los de cmadrid
    estaciones_madrid = estaciones_madrid.rename(columns={col: f"analizador_{col}" for col in ['NO2', 'SO2', 'CO', 'PM10', 'PM2_5', 'O3', 'BTX']})
    # convertimos "fecha_alta" a datetime
    estaciones_madrid["Fecha alta"] = pd.to_datetime(estaciones_madrid["Fecha alta"], format = "%d/%m/%Y")
    # elegimos el mismo formato de coordenadas que en estaciones_cmadrid y eliminamos columnas sobrantes
    estaciones_madrid.drop(columns=["CODIGO_CORTO", "COD_TIPO", "COD_VIA", "VIA_CLASE", "VIA_PAR", "VIA_NOMBRE", "NOM_TIPO", 
                                    'COORDENADA_X_ETRS89','COORDENADA_Y_ETRS89', 'LONGITUD', 'LATITUD'], inplace=True)
    # pasamos las columnas que lo necesitan a snake case
    columnas_cambio = ['CODIGO', 'ESTACION', 'DIRECCION','LONGITUD_ETRS89', 'LATITUD_ETRS89',
        'ALTITUD','Fecha alta']
    estaciones_madrid= estaciones_madrid.rename(columns ={col: col.lower().replace(" ", "_") for col in columnas_cambio})
    # renombramos columnas
    estaciones_madrid = estaciones_madrid.rename(columns={'longitud_etrs89':'coord_longitud','latitud_etrs89':'coord_latitud', "estacion": "nombre_estacion"})
    # añadimos columnas que nos faltan y que, al tratarse de Madrid tienen un valor fijo
    estaciones_madrid["municipio"] = "Madrid"
    estaciones_madrid["subarea_rural"] = "No aplica"
    estaciones_madrid["codigo_zona"] = "1"
    df_estaciones = pd.concat([estaciones_cmadrid, estaciones_madrid], ignore_index=True)
    # capitalizamos la columna tipo_estacion y rellenamos con "Fondo" los valores nulos, ya que son estaciones colocadas en parques y ese parece ser el valor típico para parques
    df_estaciones["tipo_estacion"] = df_estaciones["tipo_estacion"].fillna("fondo").str.title()
    # Rellenamos valores nulos
    df_estaciones[["analizador_NO", "analizador_PM1", "analizador_O3Q",
                    "analizador_HCT", "analizador_HNM"]] = df_estaciones[["analizador_NO", "analizador_PM1",
                                                                        "analizador_O3Q", "analizador_HCT", "analizador_HNM"]] .fillna("No")
    # convertimos todas las columnas de "analizador_" en tipo bool para eso aplicamos una lambda que mapee "Si" por True y "No" por False
    analizadores = [col for col in df_estaciones.columns if col.startswith("analizador_")]
    df_estaciones[analizadores] = df_estaciones[analizadores].apply(lambda x: x.map({"Si": True, "No": False})).astype(bool)
    df_estaciones = df_estaciones.rename(columns={"codigo":"codigo_estacion"})
    df_estaciones["codigo_estacion"] = df_estaciones["codigo_estacion"].astype(str)
    df_estaciones["codigo_municipio"] = df_estaciones["codigo_estacion"].str[2:5]
    df_municipios = df_estaciones[["codigo_municipio", "municipio"]].drop_duplicates()
    df_municipios["codigo_provincia"] = "28"
    df_estaciones.drop(columns=["municipio"], inplace=True)
    return df_estaciones, df_municipios